# Prerequisites: `.env` file

Load DataRobot credentials into your environment before you run the rest of this notebook. Create an `.env` file next to this notebook (or export variables in your shell) with at least:

```bash
DATAROBOT_ENDPOINT=YOUR_DATAROBOT_ENDPOINT
DATAROBOT_API_TOKEN=YOUR_DATAROBOT_API_TOKEN
```

Replace the placeholders with your API endpoint and token. For how routing uses these values, see [LLM configuration](llm.md).

In [ ]:
# -------------------
# 0. Load dotenv and install dependencies.
# -------------------
%load_ext dotenv
%dotenv

!uv pip install "datarobot-genai[langgraph]>=0.15.6"
!uv tool install git+https://github.com/carsongee/get-datarobot-llms.git

# Agent building blocks

## LLM

The DataRobot LLM Gateway lists the models your account can use. Run the next cell to see ids you can pass to `get_datarobot_gateway_llm`.

**Tip:** This step may take a moment depending on your network connection.

In [ ]:
# -------------------
# 1. List available LLM ids.
# -------------------
!dr get-llms

`datarobot_genai` exposes a unified LLM layer (adapters) for LangGraph, LlamaIndex, CrewAI, and NAT. For the DataRobot LLM Gateway, import `get_datarobot_gateway_llm` from the `llm` submodule of the integration you use (here: `datarobot_genai.langgraph.llm`).

In [ ]:
# -------------------
# 2. Instantiate an LLM object.
# -------------------
from datarobot_genai.langgraph.llm import get_datarobot_gateway_llm

llm = get_datarobot_gateway_llm("azure/gpt-5-nano-2025-08-07")

## Tools

You can attach tools from `datarobot_genai.drtools` to your agent. This example uses the calculator helper.

**Note:** `drtools` is still evolving; not every tool is supported for standalone use outside `drmcp`.

In [ ]:
# -------------------
# 3. Register tools.
# -------------------
from datarobot_genai.drtools.calculator import calculator
from langchain.tools import tool

tools = [tool(calculator)]

## Agent

Define the graph the same way you would in plain LangGraph: same primitives (`StateGraph`, nodes, edges). That makes it straightforward to reuse or adapt existing examples.

In [ ]:
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END, MessagesState, START, StateGraph

# -------------------
# 4.1. Agent prompt.
# -------------------
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. {chat_history}"),
        ("user", "{topic}"),
    ]
)


def graph_factory(*args):
    # -------------------
    # 4.2. ReAct agents as nodes.
    # -------------------
    research_agent = create_agent(llm, tools)
    writer_agent = create_agent(llm, [])

    # -------------------
    # 4.3. Wire the graph.
    # -------------------
    graph = StateGraph(MessagesState)
    graph.add_node("research", research_agent)
    graph.add_node("writer", writer_agent)
    graph.add_edge(START, "research")
    graph.add_edge("research", "writer")
    graph.add_edge("writer", END)
    return graph

# Wrap the graph for DataRobot

Each supported integration exposes a helper that turns your graph factory and prompt template into a DataRobot agent class you can run behind AG-UI.

In [ ]:
# -------------------
# 5. Convert to a DataRobot agent class.
# -------------------
from datarobot_genai.langgraph.agent import datarobot_agent_class_from_langgraph

MyAgent = datarobot_agent_class_from_langgraph(graph_factory, prompt_template)

# Run the agent

The next cell streams AG-UI events from `invoke` so you can inspect the run lifecycle, tool calls, and text.

In [ ]:
# -------------------
# 5. Action!
# -------------------
from ag_ui.core import UserMessage
from ag_ui.core import RunAgentInput
import uuid

agent = MyAgent()

run_input = RunAgentInput(
    thread_id=str(uuid.uuid4()),
    run_id=str(uuid.uuid4()),
    messages=[UserMessage(id=str(uuid.uuid4()), role="user", content="Calculate 2+2 and outputn result immediately")],
    state=[],
    tools=[],
    context=[],
    forwardedProps=None
)

async for event, _interactions, usage in agent.invoke(run_input):
    print(event)